In [ ]:
pip install simpy

In [ ]:
import simpy, random

# Часть 1 - Демонстрационная модель




In [ ]:
lmbd1 = 20 # звонков в час
lmbd2 = 6  # звонков в час
mu1 = 1 / (3 / 60) # звонков в час
mu2 = 1 / (7 / 60) # звонков в час

In [ ]:
N = 2 # число операторов
K = 2 # число 'мест для ожидания'

In [ ]:
T = 1 # время моделирования (час)

Функция генерации звонков от абонентов тарифа Смарт

In [ ]:
def type1(env, call_center):
    i = 0
    while True:
        i += 1
        at= random.expovariate(lmbd1)
        yield env.timeout(at)
        st = random.expovariate(mu1)
        env.process(service(env, st, call_center, f'Абонент_Смарт_{i}'))

Функция генерации звонков от абонентов тарифа Базовый

In [ ]:
def type2(env, call_center):
    i = 0
    while True:
        at= random.expovariate(lmbd2)
        yield env.timeout(at)
        st = random.expovariate(mu2)
        env.process(service(env, st, call_center, f'Абонент_Базовый_{i}'))

Функция обслуживания абонентов

In [ ]:
def service(env, st, call_center, name):
    print(f'{env.now:.2f}: {name} позвонил')
    if call_center.count < N:
        req = call_center.request()
        yield req
        print(f'{env.now:.2f}: {name} соединился с оператором')
        yield env.timeout(st)
        print(f'{env.now:.2f}: {name} закончил звонок')
        call_center.release(req)

    elif len(call_center.queue):
        req = call_center.request()
        print(f'{env.now:.2f}: {name} будет ждать в очереди')
        yield req
        print(f'{env.now:.2f}: {name} дождался оператора')
        yield env.timeout(st)
        print(f'{env.now:.2f}: {name} закончил звонок')
        call_center.release(req)

    else:
        print(f'{env.now:.2f}: {name} сбросил звонок,т.к все операторы заняты и нет мест для ожиданий')

In [ ]:
env = simpy.Environment()
call_center = simpy.Resource(env, capacity = N)
env.process(type1(env, call_center))
env.process(type2(env, call_center))
a = str(env.run(until = T))

0.04: Абонент_Смарт_1 позвонил
0.04: Абонент_Смарт_1 соединился с оператором
0.04: Абонент_Смарт_2 позвонил
0.04: Абонент_Смарт_2 соединился с оператором
0.04: Абонент_Смарт_2 закончил звонок
0.05: Абонент_Смарт_3 позвонил
0.05: Абонент_Смарт_3 соединился с оператором
0.05: Абонент_Смарт_1 закончил звонок
0.06: Абонент_Смарт_4 позвонил
0.06: Абонент_Смарт_4 соединился с оператором
0.08: Абонент_Смарт_4 закончил звонок
0.11: Абонент_Смарт_3 закончил звонок
0.16: Абонент_Базовый_0 позвонил
0.16: Абонент_Базовый_0 соединился с оператором
0.18: Абонент_Смарт_5 позвонил
0.18: Абонент_Смарт_5 соединился с оператором
0.21: Абонент_Смарт_5 закончил звонок
0.24: Абонент_Базовый_0 позвонил
0.24: Абонент_Базовый_0 соединился с оператором
0.24: Абонент_Базовый_0 закончил звонок
0.26: Абонент_Смарт_6 позвонил
0.26: Абонент_Смарт_6 соединился с оператором
0.26: Абонент_Смарт_6 закончил звонок
0.32: Абонент_Базовый_0 закончил звонок
0.32: Абонент_Смарт_7 позвонил
0.32: Абонент_Смарт_7 соединился с оп

# Часть 2 - Расчет оптимальной конфигурации системы


Функция генерации звонков от абонентов тарифа Смарт

In [ ]:
def type1(env, call_center):
    while True:
        at = random.expovariate(lmbd1)
        yield env.timeout(at)
        st = random.expovariate(mu1)
        env.process(service(env, st, call_center))

Функция генерации звонков от абонентов тарифа Базовый

In [ ]:
def type2(env, call_center):
    while True:
        at = random.expovariate(lmbd2)
        yield env.timeout(at)
        st = random.expovariate(mu2)
        env.process(service(env, st, call_center))

Функция обслуживания абонентов

In [ ]:
def service(env, st, call_center):
    global total_number
    global drop_number
    total_number += 1

    if call_center.count < N:
        req = call_center.request()
        yield req
        yield env.timeout(st)
        call_center.release(req)

    elif len(call_center.queue) < K:
        req = call_center.request()
        yield req
        yield env.timeout(st)
        call_center.release(req)

    else:
        drop_number += 1

Поиск оптимальной конфигурации

In [ ]:
def simulation(N, K):
    env = simpy.Environment()
    call_center = simpy.Resource(env, N)
    env.process(type1(env, call_center))
    env.process(type2(env, call_center))
    env.run(until = 1000)

In [ ]:
flag = False
N = 1

while not flag:
    for K in range(8):
        drop_number = 0
        total_number = 0
        simulation(N,K)
        if drop_number / total_number < 0.01:
            flag = True
            print(f'N = {N}, K = {K}')
            print(f'Вероятность того, что звонок будет сброшен: {drop_number / total_number:.3f}')
            break
    N += 1

N = 3, K = 6
Вероятность того, что звонок будет сброшен: 0.007


# КАКИЕ МОЖНО СДЕЛАТЬ ВЫВОДЫ НА ОСНОВАНИИ ПОСТРОЕНОЙ МОДЕЛИ:

> Размер оптимальных вводных:  N = 3 , K >= 2 , т.к при данных показателях числа операторов и числа 'мест для ожидания', можно достигнуть наименьшего результата потери < 0.01, при высокой эффективности работы колл-центра


